In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.gold.dim_customer AS
SELECT
    customer_id AS customer_key,
    customer_fname AS first_name,
    customer_lname AS last_name,
    customer_segment,
    customer_city,
    customer_state,
    customer_country,
    customer_zipcode
FROM workspace.silver.supply_chain_orders
QUALIFY row_number() OVER (PARTITION BY customer_id ORDER BY order_date_dateorders DESC) = 1
""")

spark.sql("""
CREATE OR REPLACE TABLE workspace.gold.dim_product AS
SELECT
    product_card_id AS product_key,
    product_name,
    product_category_id AS category_id,
    category_name,
    department_id,
    department_name,
    product_price AS list_price,
    product_status
FROM workspace.silver.supply_chain_orders
QUALIFY row_number() OVER (PARTITION BY product_card_id ORDER BY order_date_dateorders DESC) = 1
""")

spark.sql("""
CREATE OR REPLACE TABLE workspace.gold.dim_geography AS
SELECT DISTINCT
    md5(concat_ws('|', order_city, order_state, order_country, order_region, market)) AS geography_key,
    order_city AS city,
    order_state AS state,
    order_country AS country,
    order_region AS region,
    market
FROM workspace.silver.supply_chain_orders
""")

spark.sql("""
CREATE OR REPLACE TABLE workspace.gold.dim_shipping_mode AS
SELECT DISTINCT
    md5(shipping_mode) AS shipping_mode_key,
    shipping_mode
FROM workspace.silver.supply_chain_orders
""")

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.gold.dim_date AS
WITH bounds AS (
    SELECT
        CAST(MIN(order_date_dateorders) AS DATE) AS first_day,
        CAST(MAX(shipping_date_dateorders) AS DATE) AS last_day
    FROM workspace.silver.supply_chain_orders
),
spine AS (
    SELECT explode(sequence(first_day, last_day, INTERVAL 1 DAY)) AS date_day
    FROM bounds
)
SELECT
    CAST(date_format(date_day, 'yyyyMMdd') AS INT) AS date_key,
    date_day,
    year(date_day) AS calendar_year,
    quarter(date_day) AS calendar_quarter,
    month(date_day) AS calendar_month,
    date_format(date_day, 'MMM') AS month_name,
    date_format(date_day, 'yyyy-MM') AS year_month,
    date_format(date_day, 'EEE') AS day_name,
    date_format(date_day, 'EEE') IN ('Sat', 'Sun') AS is_weekend
FROM spine
""")

spark.sql("""
CREATE OR REPLACE TABLE workspace.gold.fct_order_items AS
SELECT
    order_item_id AS order_item_key,
    order_id,
    customer_id AS customer_key,
    product_card_id AS product_key,
    md5(concat_ws('|', order_city, order_state, order_country, order_region, market)) AS geography_key,
    md5(shipping_mode) AS shipping_mode_key,
    CAST(date_format(order_date_dateorders, 'yyyyMMdd') AS INT) AS date_key,
    CAST(date_format(shipping_date_dateorders, 'yyyyMMdd') AS INT) AS shipping_date_key,
    `type` AS payment_type,
    order_status,
    delivery_status,
    order_item_quantity AS quantity,
    order_item_product_price AS unit_price,
    sales,
    order_item_discount AS discount,
    order_item_discount_rate AS discount_rate,
    order_item_total,
    order_profit_per_order AS profit,
    order_item_profit_ratio AS profit_ratio,
    days_for_shipping_real AS shipping_days_real,
    days_for_shipment_scheduled AS shipping_days_scheduled,
    days_for_shipping_real - days_for_shipment_scheduled AS shipping_delay_days,
    (delivery_status = 'Late delivery') AS is_late
FROM workspace.silver.supply_chain_orders
""")

In [0]:
display(spark.sql("""
SELECT 'silver rows' AS check_name, COUNT(*) AS value FROM workspace.silver.supply_chain_orders
UNION ALL SELECT 'fact rows', COUNT(*) FROM workspace.gold.fct_order_items
UNION ALL SELECT 'duplicate order_item_key', COUNT(*) - COUNT(DISTINCT order_item_key) FROM workspace.gold.fct_order_items
UNION ALL SELECT 'fact rows with no order date', COUNT(*) FROM workspace.gold.fct_order_items WHERE date_key IS NULL
UNION ALL SELECT 'orphan customers', COUNT(*) FROM workspace.gold.fct_order_items f
    LEFT JOIN workspace.gold.dim_customer d ON f.customer_key = d.customer_key WHERE d.customer_key IS NULL
UNION ALL SELECT 'orphan products', COUNT(*) FROM workspace.gold.fct_order_items f
    LEFT JOIN workspace.gold.dim_product d ON f.product_key = d.product_key WHERE d.product_key IS NULL
UNION ALL SELECT 'orphan geographies', COUNT(*) FROM workspace.gold.fct_order_items f
    LEFT JOIN workspace.gold.dim_geography d ON f.geography_key = d.geography_key WHERE d.geography_key IS NULL
UNION ALL SELECT 'orphan dates', COUNT(*) FROM workspace.gold.fct_order_items f
    LEFT JOIN workspace.gold.dim_date d ON f.date_key = d.date_key WHERE d.date_key IS NULL
"""))

In [0]:
display(spark.sql("""
SELECT g.market, ROUND(SUM(f.profit), 0) AS total_profit
FROM workspace.gold.fct_order_items f
JOIN workspace.gold.dim_geography g ON f.geography_key = g.geography_key
GROUP BY g.market
ORDER BY total_profit DESC
"""))
